# 03 — Protocol Loader

`src/protocol_loader.py` downloads a protocol `.docx` from Drive, extracts body paragraphs + table cells, loads the companion knowledge Google Doc if it exists, and returns all components separately for `build_system_prompt()`.

## Why tables must be extracted separately

`doc.paragraphs` from python-docx only covers body paragraphs. Buffer recipes almost always live in tables and are **not** included. `doc.tables` must be iterated explicitly.

In [ ]:
import sys, asyncio, io
sys.path.insert(0, '..')

from src.protocol_loader import extract_docx_text, load_protocol
from src.google_client import list_protocols
print('Imports OK')

## 1. Extract text from a .docx file

In [ ]:
from src.google_client import download_docx

protocols = asyncio.run(list_protocols())
if protocols:
    p = protocols[0]
    raw = asyncio.run(download_docx(p['id']))
    text = extract_docx_text(raw)
    print(f"Protocol: {p['name']}")
    print(f"Extracted: {len(text)} chars")
    print('---')
    print(text[:1000])
else:
    print('No protocols in Drive.')

## 2. Full load_protocol() — returns all components

This is what `ProtocolSession.create()` calls internally.

In [ ]:
if protocols:
    p = protocols[0]
    protocol_text, companion_text, protocol_name, protocol_version, companion_doc_id = \
        asyncio.run(load_protocol(p['id'], p['name'], p.get('modifiedTime','')))

    print(f'Protocol name:    {protocol_name}')
    print(f'Protocol version: {protocol_version}')
    print(f'Protocol text:    {len(protocol_text)} chars')
    print(f'Companion text:   {len(companion_text)} chars ({"loaded" if companion_text else "empty"})')
    print(f'Companion doc id: {companion_doc_id}')

## 3. System prompt preview

See how the combined context looks when injected into the Protocol Expert system prompt.

In [ ]:
from src.claude_client import build_system_prompt

if protocols:
    prompt = build_system_prompt(
        protocol_text=protocol_text,
        companion_text=companion_text if companion_text.strip() else None,
        protocol_name=protocol_name,
        protocol_version=protocol_version,
    )
    print(f'System prompt length: {len(prompt)} chars')
    print('--- First 1500 chars ---')
    print(prompt[:1500])